<a href="https://colab.research.google.com/github/Shivam6901/Software-Engineering-Project/blob/main/SOAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL 1] INSTALL BINARIES & PYTHON PACKAGES
!apt-get update -qq && apt-get install -y -qq nmap iptables
!pip install -q --upgrade scapy pandas plotly streamlit pyngrok google-generativeai

from pyngrok import ngrok
print("Installing Ngrok binary...")
ngrok.install_ngrok()
print("✅ Binaries and Python packages installed successfully.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libip4tc2:amd64.
(Reading database ... 128200 files and directories currently installed.)
Preparing to unpack .../0-libip4tc2_1.8.10-3ubuntu2_amd64.deb ...
Unpacking libip4tc2:amd64 (1.8.10-3ubuntu2) ...
Selecting previously unselected package libip6tc2:amd64.
Preparing to unpack .../1-libip6tc2_1.8.10-3ubuntu2_amd64.deb ...
Unpacking libip6tc2:amd64 (1.8.10-3ubuntu2) ...
Selecting previously unselected package libnfnetlink0:amd64.
Preparing to unpack .../2-libnfnetlink0_1.0.2-2build1_amd64.deb ...
Unpacking libnfnetlink0:amd64 (1.0.2-2build1) ...
Selecting previously unselected package libnetfilter-conntrack3:amd64.
Preparing to unpack .../3-libnetfilter-conntrack3_1.0.9-6build1_amd64.deb ...
Unpacking libnetfilter-conntrack3:amd64 (1.0.9-6build1) ...
Selectin

In [ ]:
# [CELL 2] GENERATE BENCHMARK PCAP (NORMAL TRAFFIC + DDOS FLOOD)
import random
import time
from scapy.all import IP, TCP, UDP, Ether, wrpcap

def generate_benchmark_pcap(filename="test_traffic.pcap", num_normal_packets=1000, num_ddos_packets=5000, attacker_ip="192.168.1.105", victim_ip="10.0.0.1", ddos_duration_seconds=5):
    packets = []
    start_time = time.time()

    # Normal background noise
    for _ in range(num_normal_packets):
        src_ip = f"192.168.1.{random.randint(2, 254)}"
        dst_ip = f"10.0.0.{random.randint(2, 254)}"
        sport = random.randint(1024, 65535)
        dport = random.randint(80, 443)
        pkt = Ether()/IP(src=src_ip, dst=dst_ip)/(TCP(sport=sport, dport=dport) if random.random() < 0.7 else UDP(sport=sport, dport=dport))
        pkt.time = start_time + random.uniform(0, ddos_duration_seconds + 10)
        packets.append(pkt)

    # Volumetric DDoS burst (From 192.168.1.105)
    for _ in range(num_ddos_packets):
        ddos_pkt_time = start_time + random.uniform(2, 2 + ddos_duration_seconds)
        sport = random.randint(1024, 65535)
        dport = 80
        pkt = Ether()/IP(src=attacker_ip, dst=victim_ip)/(TCP(sport=sport, dport=dport, flags="S") if random.random() < 0.6 else UDP(sport=sport, dport=dport))
        pkt.time = ddos_pkt_time
        packets.append(pkt)

    packets.sort(key=lambda x: x.time)
    wrpcap(filename, packets)
    print(f"✅ Benchmark PCAP '{filename}' created with {len(packets):,} packets.")

generate_benchmark_pcap()

✅ Benchmark PCAP 'test_traffic.pcap' created with 6,000 packets.


In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import datetime
import subprocess
import os
import time
import random
import shlex
import threading
import collections
import urllib.request
from scapy.all import sniff, AsyncSniffer, get_if_list, IP, TCP, UDP, ICMP, Ether, PcapReader

# Gemini API Integration
try:
    import google.generativeai as genai
    _HAS_GEMINI = True
except ImportError:
    _HAS_GEMINI = False

# --- Page Setup ---
st.set_page_config(layout="wide", page_title="SOC Command Center // SOAR Telemetry", page_icon="🛡️")

# --- Refined High-End SOC Glassmorphism Styling ---
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;500;600;700&family=Inter:wght@400;500;600;700&display=swap');

    :root {
        --bg-base: #06090e;
        --bg-card: #0b111c;
        --bg-subpanel: #111a2a;
        --border-subtle: #182337;
        --border-accent: #253654;
        --text-main: #f8fafc;
        --text-muted: #94a3b8;
        --accent-blue: #38bdf8;
        --accent-crimson: #f43f5e;
        --accent-emerald: #10b981;
        --accent-amber: #fbbf24;
    }

    html, body, [class*="css"] {
        font-family: 'Inter', -apple-system, sans-serif !important;
        background-color: var(--bg-base) !important;
        color: var(--text-main) !important;
    }
    .stApp { background-color: var(--bg-base); }

    .soc-header {
        display: flex;
        justify-content: space-between;
        align-items: center;
        padding: 12px 18px;
        margin-bottom: 16px;
        background: linear-gradient(90deg, #0d1524 0%, #080c14 100%);
        border: 1px solid var(--border-subtle);
        border-radius: 6px;
    }
    .soc-title {
        font-family: 'JetBrains Mono', monospace;
        font-size: 14px;
        font-weight: 700;
        letter-spacing: 0.08em;
        color: var(--text-main);
    }
    .soc-badge {
        font-family: 'JetBrains Mono', monospace;
        font-size: 11px;
        padding: 4px 10px;
        border-radius: 4px;
        font-weight: 600;
        letter-spacing: 0.05em;
    }
    .badge-live {
        background: rgba(16, 185, 129, 0.12);
        border: 1px solid rgba(16, 185, 129, 0.4);
        color: #34d399;
    }
    .badge-alert {
        background: rgba(244, 63, 94, 0.15);
        border: 1px solid rgba(244, 63, 94, 0.5);
        color: #fb7185;
    }

    div[data-testid="stMetric"] {
        background: linear-gradient(180deg, #0d1525 0%, #090e18 100%);
        border: 1px solid var(--border-subtle);
        border-top: 2px solid var(--accent-blue);
        border-radius: 6px;
        padding: 12px 16px;
        box-shadow: 0 4px 12px rgba(0, 0, 0, 0.25);
    }
    div[data-testid="stMetricValue"] {
        font-family: 'JetBrains Mono', monospace !important;
        color: var(--text-main) !important;
        font-size: 22px !important;
        font-weight: 700 !important;
    }
    div[data-testid="stMetricLabel"] {
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 10.5px !important;
        color: var(--text-muted) !important;
        text-transform: uppercase;
        letter-spacing: 0.06em;
    }

    .panel-header {
        font-family: 'JetBrains Mono', monospace;
        font-size: 11px;
        font-weight: 600;
        text-transform: uppercase;
        letter-spacing: 0.08em;
        color: var(--accent-blue);
        margin: 18px 0 8px 0;
        display: flex;
        align-items: center;
        gap: 8px;
    }

    .soc-term {
        background-color: #04070c;
        border: 1px solid var(--border-accent);
        border-left: 3px solid var(--accent-blue);
        border-radius: 4px;
        padding: 12px 14px;
        font-family: 'JetBrains Mono', monospace;
        font-size: 11.5px;
        color: #e2e8f0;
        line-height: 1.5;
        white-space: pre-wrap;
        word-break: break-all;
    }
    .soc-term-amber {
        border-left-color: var(--accent-amber);
        background: #060910;
    }

    .status-strip-red {
        background: linear-gradient(90deg, rgba(244, 63, 94, 0.15) 0%, rgba(244, 63, 94, 0.03) 100%);
        border: 1px solid rgba(244, 63, 94, 0.4);
        color: #fb7185;
        font-family: 'JetBrains Mono', monospace;
        font-size: 11.5px;
        font-weight: 600;
        letter-spacing: 0.06em;
        text-transform: uppercase;
        padding: 10px 14px;
        border-radius: 4px;
        margin-bottom: 12px;
    }
    .status-strip-green {
        background: linear-gradient(90deg, rgba(16, 185, 129, 0.12) 0%, rgba(16, 185, 129, 0.02) 100%);
        border: 1px solid rgba(16, 185, 129, 0.35);
        color: #34d399;
        font-family: 'JetBrains Mono', monospace;
        font-size: 11.5px;
        font-weight: 600;
        letter-spacing: 0.06em;
        text-transform: uppercase;
        padding: 10px 14px;
        border-radius: 4px;
        margin-bottom: 12px;
    }

    .stButton>button {
        background: linear-gradient(180deg, #162238 0%, #0d1524 100%) !important;
        color: var(--text-main) !important;
        border: 1px solid var(--border-accent) !important;
        border-radius: 4px !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 11px !important;
        font-weight: 600 !important;
        text-transform: uppercase;
        letter-spacing: 0.06em;
        padding: 9px 16px !important;
        transition: all 0.2s ease;
    }
    .stButton>button:hover {
        border-color: var(--accent-blue) !important;
        box-shadow: 0 0 10px rgba(56, 189, 248, 0.2) !important;
        color: #ffffff !important;
    }

    section[data-testid="stSidebar"] {
        background-color: #070b12 !important;
        border-right: 1px solid var(--border-subtle) !important;
    }
    .stDataFrame {
        border: 1px solid var(--border-subtle) !important;
        border-radius: 4px;
    }
</style>
""", unsafe_allow_html=True)

# --- Global Session State ---
if "live_records" not in st.session_state:
    st.session_state.live_records = []
if "is_capturing" not in st.session_state:
    st.session_state.is_capturing = False
if "packet_buffer" not in st.session_state:
    st.session_state.packet_buffer = collections.deque(maxlen=10000)
if "nmap_output" not in st.session_state:
    st.session_state.nmap_output = None
if "mitigation_log" not in st.session_state:
    st.session_state.mitigation_log = []
if "capture_start_time" not in st.session_state:
    st.session_state.capture_start_time = None
if "uploaded_df" not in st.session_state:
    st.session_state.uploaded_df = None
if "async_sniffer" not in st.session_state:
    st.session_state.async_sniffer = None

# Auto-Discovery for Gemini
if "gemini_model" not in st.session_state:
    api_key = os.environ.get("GOOGLE_API_KEY")
    if _HAS_GEMINI and api_key:
        try:
            genai.configure(api_key=api_key)
            supported = [m.name for m in genai.list_models() if "generateContent" in m.supported_generation_methods]
            chosen = None
            for cand in ["models/gemini-2.5-flash", "models/gemini-2.0-flash", "models/gemini-1.5-flash", "models/gemini-pro"]:
                if cand in supported:
                    chosen = cand
                    break
            if not chosen and supported:
                chosen = supported[0]
            st.session_state.gemini_model = genai.GenerativeModel(chosen) if chosen else None
            st.session_state.model_tag = chosen.replace("models/", "") if chosen else "SOC Rule Core"
        except Exception:
            st.session_state.gemini_model = None
            st.session_state.model_tag = "SOC Rule Core"
    else:
        st.session_state.gemini_model = None
        st.session_state.model_tag = "SOC Rule Core"

# --- Packet Parsing Helper ---
def parse_packet_data(pkt):
    t = float(pkt.time)
    src = pkt[IP].src if pkt.haslayer(IP) else (pkt[Ether].src if pkt.haslayer(Ether) else "Unknown")
    dst = pkt[IP].dst if pkt.haslayer(IP) else (pkt[Ether].dst if pkt.haslayer(Ether) else "Unknown")
    ttl = pkt[IP].ttl if pkt.haslayer(IP) else "N/A"
    proto, sport, dport = "Other", 0, 80

    if pkt.haslayer(TCP):
        proto, sport, dport = "TCP", pkt[TCP].sport, pkt[TCP].dport
    elif pkt.haslayer(UDP):
        proto, sport, dport = "UDP", pkt[UDP].sport, pkt[UDP].dport
    elif pkt.haslayer(ICMP):
        proto = "ICMP"

    raw_bytes = bytes(pkt)
    return {
        'Packet ID': 0,
        'Timestamp': t,
        'Time': datetime.datetime.fromtimestamp(t).strftime('%H:%M:%S.%f')[:-3],
        'Source IP': src,
        'Source Port': sport,
        'Destination IP': dst,
        'Destination Port': dport,
        'Protocol': proto,
        'Length': len(pkt),
        'TTL': ttl,
        'Hex': " ".join(f"{b:02x}" for b in raw_bytes[:64]),
        'Ascii': "".join(chr(b) if 32 <= b <= 126 else "." for b in raw_bytes[:64]),
        'Summary': pkt.summary(),
        'Location': "Private Subnet" if src.startswith(("192.168.", "10.", "172.16.", "127.")) else "Public Node"
    }

# Callback for AsyncSniffer
def _scapy_async_callback(pkt):
    try:
        row = parse_packet_data(pkt)
        if row:
            st.session_state.packet_buffer.append(row)
    except Exception:
        pass

# Background Traffic Generator: Sends real queries to keep the VM network interface active
def vm_network_heartbeat_worker():
    while st.session_state.get('is_capturing', False):
        try:
            # Generate real outbound network requests (DNS & HTTP)
            urllib.request.urlopen("https://1.1.1.1", timeout=0.8)
            subprocess.run(["ping", "-c", "1", "-W", "1", "8.8.8.8"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception:
            pass
        time.sleep(0.4)

def start_real_capture():
    # Detect best active interface
    ifaces = get_if_list()
    target_iface = "eth0" if "eth0" in ifaces else (ifaces[0] if ifaces else None)
    try:
        sniffer = AsyncSniffer(prn=_scapy_async_callback, store=False, iface=target_iface)
        sniffer.start()
        st.session_state.async_sniffer = sniffer
        # Kick off background heartbeat thread so packets actually cross the wire
        threading.Thread(target=vm_network_heartbeat_worker, daemon=True).start()
    except Exception:
        # Fallback to general capture
        sniffer = AsyncSniffer(prn=_scapy_async_callback, store=False)
        sniffer.start()
        st.session_state.async_sniffer = sniffer

def stop_real_capture():
    if st.session_state.get("async_sniffer"):
        try:
            st.session_state.async_sniffer.stop()
        except Exception:
            pass
        st.session_state.async_sniffer = None

# Simulated Attack Benchmark Generator
def simulated_traffic_tick(elapsed_time):
    t = time.time()
    for _ in range(random.randint(15, 25)):
        src = f"192.168.1.{random.choice([10, 16, 79, 131, 157, 182, 195, 215])}"
        sport = random.randint(1024, 65535)
        proto = "TCP" if random.random() < 0.75 else "UDP"
        dport = random.choice([80, 443, 53, 22])
        st.session_state.packet_buffer.append({
            'Packet ID': 0,
            'Timestamp': t + random.uniform(0, 0.4),
            'Time': datetime.datetime.fromtimestamp(t).strftime('%H:%M:%S.%f')[:-3],
            'Source IP': src, 'Source Port': sport,
            'Destination IP': "10.0.0.1", 'Destination Port': dport,
            'Protocol': proto, 'Length': random.randint(64, 1200),
            'TTL': 64, 'Hex': "45 00 00 3c 1a ...", 'Ascii': "E..<..",
            'Summary': f"{proto} {src}:{sport} -> 10.0.0.1:{dport}",
            'Location': "Private Subnet"
        })

    if elapsed_time > 4.0:
        for _ in range(random.randint(120, 180)):
            sport = random.randint(1024, 65535)
            st.session_state.packet_buffer.append({
                'Packet ID': 0,
                'Timestamp': t + random.uniform(0, 0.4),
                'Time': datetime.datetime.fromtimestamp(t).strftime('%H:%M:%S.%f')[:-3],
                'Source IP': "192.168.1.105", 'Source Port': sport,
                'Destination IP': "10.0.0.1", 'Destination Port': 80,
                'Protocol': "TCP", 'Length': 64, 'TTL': 64,
                'Hex': "45 00 00 28 00 01 00 00 40 06 7c b2 c0 a8 01 69 0a 00 00 01",
                'Ascii': "E..(...@..|...i....",
                'Summary': f"TCP 192.168.1.105:{sport} -> 10.0.0.1:80 [SYN]",
                'Location': "Private Subnet"
            })

# --- Shell Operations ---
def run_direct_nmap(target_ip, flags):
    cmd_list = ["sudo", "nmap", *shlex.split(flags), target_ip]
    cmd_display = f"sudo nmap {flags} {target_ip}"
    try:
        res = subprocess.run(cmd_list, capture_output=True, text=True, timeout=20)
        out = res.stdout if res.stdout else res.stderr
        return cmd_display, out if out.strip() else "[!] Probe filtered by ingress firewall."
    except subprocess.TimeoutExpired:
        return cmd_display, f"[!] Probe timed out for target host {target_ip}."
    except Exception as e:
        return cmd_display, f"Command execution error: {e}"

def enforce_iptables_drop(ip_address):
    try:
        subprocess.run(["sudo", "iptables", "-A", "INPUT", "-s", ip_address, "-j", "DROP"], capture_output=True, check=True)
        return f"[KERNEL] Drop rule appended to INPUT chain for {ip_address}."
    except Exception as e:
        return f"[ERROR] Kernel insertion failed: {e}"

def generate_ai_threat_summary(nmap_output_text):
    fallback_text = (
        "• Host State: Target appears firewalled or silently dropping active recon probes.\n"
        "• Attack Surface: No responsive TCP listening ports identified in Fast Sweep.\n"
        "• SOC Action: Enforce perimeter ingress filtering via iptables DROP and inspect upstream flows."
    )
    model = st.session_state.get('gemini_model')
    tag = st.session_state.get('model_tag', 'SOC Heuristics')
    if model:
        try:
            prompt = (
                "You are a tier-3 SOC analyst. Analyze this Nmap recon report. "
                "Provide exactly 3 concise, formal bullet points on: (1) Host exposure, "
                "(2) Potential vulnerability exposure, and (3) Recommended containment actions.\n\n"
                f"```\n{nmap_output_text}\n```"
            )
            res = model.generate_content(prompt)
            summary = res.text
        except Exception:
            summary = fallback_text
    else:
        summary = fallback_text

    return f"""
<div class='soc-term soc-term-amber'>
<span style='color:#fbbf24; font-weight:600;'>[INCIDENT TRIAGE REPORT // {tag.upper()}]</span><br>
----------------------------------------------------------------------<br>
{summary}
</div>
"""

# --- Sidebar Pipeline Controls ---
st.sidebar.markdown("<div style='font-family:JetBrains Mono; font-size:11px; font-weight:700; color:#64748b; letter-spacing:0.1em; margin-bottom:8px;'>TELEMETRY PIPELINE</div>", unsafe_allow_html=True)
mode = st.sidebar.radio("Acquisition Engine", [
    "Simulated Attack Benchmark",
    "Mini-Wireshark Live Sniffer",
    "Upload Forensic PCAP"
])

ddos_threshold = st.sidebar.slider("DDoS Volumetric Threshold (PPS)", 20, 1000, 130, 10)
defense_mode = st.sidebar.radio("Defense Policy", [
    "Monitor Mode (Audit Only)",
    "Active Containment (Auto-Drop Attacker)"
])

uploaded_file = None
if mode == "Upload Forensic PCAP":
    uploaded_file = st.sidebar.file_uploader("Upload PCAP capture", type=["pcap", "pcapng"])

st.sidebar.markdown("<hr style='border-color:#182337; margin:16px 0;'>", unsafe_allow_html=True)
if st.sidebar.button("Flush Active Firewall Rules"):
    try:
        subprocess.run(["sudo", "iptables", "-F"], capture_output=True, check=True)
        st.session_state.mitigation_log = []
        st.sidebar.success("Firewall rules flushed.")
    except Exception as e:
        st.sidebar.error(f"Flush error: {e}")

# Drain Background Packet Buffer
while st.session_state.packet_buffer:
    pkt_data = st.session_state.packet_buffer.popleft()
    pkt_data['Packet ID'] = len(st.session_state.live_records) + 1
    st.session_state.live_records.append(pkt_data)

if len(st.session_state.live_records) > 20000:
    st.session_state.live_records = st.session_state.live_records[-15000:]

# --- Capture Lifecycle Controls ---
if mode != "Upload Forensic PCAP":
    c1, c2, c3, _ = st.columns([1.5, 1.5, 1.2, 3.8])
    with c1:
        if not st.session_state.is_capturing:
            if st.button("Start Acquisition", use_container_width=True):
                st.session_state.is_capturing = True
                st.session_state.capture_start_time = time.time()
                if mode == "Mini-Wireshark Live Sniffer":
                    start_real_capture()
                st.rerun()
    with c2:
        if st.session_state.is_capturing:
            if st.button("Stop & Freeze Triage", use_container_width=True):
                st.session_state.is_capturing = False
                if mode == "Mini-Wireshark Live Sniffer":
                    stop_real_capture()
                st.rerun()
    with c3:
        if st.button("Clear Buffer", use_container_width=True):
            st.session_state.live_records = []
            st.session_state.mitigation_log = []
            st.rerun()

    if st.session_state.is_capturing and mode == "Simulated Attack Benchmark":
        elapsed = time.time() - (st.session_state.capture_start_time or time.time())
        simulated_traffic_tick(elapsed)

    raw_df = pd.DataFrame(st.session_state.live_records)

else:
    if uploaded_file is not None:
        if st.session_state.uploaded_df is None:
            with open("temp_upload.pcap", "wb") as f:
                f.write(uploaded_file.getbuffer())
            rows = []
            with PcapReader("temp_upload.pcap") as reader:
                for idx, pkt in enumerate(reader):
                    r = parse_packet_data(pkt)
                    if r:
                        r['Packet ID'] = idx + 1
                        rows.append(r)
            st.session_state.uploaded_df = pd.DataFrame(rows)
            st.sidebar.success(f"PCAP Loaded ({len(rows):,} packets)")
        raw_df = st.session_state.uploaded_df
    else:
        st.session_state.uploaded_df = None
        raw_df = pd.DataFrame()

# ==============================================================================
# UNIFIED SOC TELEMETRY & INCIDENT RESPONSE CONSOLE
# ==============================================================================
if not raw_df.empty:
    raw_df['Timestamp_dt'] = pd.to_datetime(raw_df['Timestamp'], unit='s')

    packet_rates = raw_df.set_index('Timestamp_dt').resample('1s').size().rename('Packet Rate').reset_index()
    packet_rates['Packet Rate'] = packet_rates['Packet Rate'].rolling(window=2, min_periods=1).mean().astype(int)

    attack_windows = packet_rates[packet_rates['Packet Rate'] > ddos_threshold]
    is_attack = not attack_windows.empty

    top_sources = raw_df['Source IP'].value_counts().reset_index()
    top_sources.columns = ['Source IP', 'Packets']
    top_sources['Share'] = ((top_sources['Packets'] / len(raw_df)) * 100).round(1).astype(str) + '%'
    top_sources['Classification'] = top_sources['Packets'].apply(
        lambda x: 'CRITICAL' if x > (ddos_threshold * 2) else ('SUSPICIOUS' if x > ddos_threshold else 'NORMAL')
    )

    top_attacker_ip = top_sources.iloc[0]['Source IP'] if not top_sources.empty else None

    # Automated Containment Action (Active Defense Policy)
    if is_attack and "Active" in defense_mode and top_attacker_ip:
        if not any(entry['Attacking IP'] == top_attacker_ip for entry in st.session_state.mitigation_log):
            kernel_log_msg = enforce_iptables_drop(top_attacker_ip)
            st.session_state.mitigation_log.append({
                'Attacking IP': top_attacker_ip,
                'Peak Rate': f"{int(packet_rates['Packet Rate'].max())} PPS",
                'Detection Time': datetime.datetime.now().strftime('%H:%M:%S'),
                'Enforcement Action': "iptables -A INPUT -j DROP",
                'Kernel Status': kernel_log_msg
            })

    # Header Bar Status
    status_badge = "badge-alert" if is_attack else "badge-live"
    status_label = "CRITICAL BREACH // CONTAINMENT ENGAGED" if (is_attack and "Active" in defense_mode) else ("ANOMALY DETECTED // AUDITING" if is_attack else "SYSTEM STATUS: OPTIMAL")

    st.markdown(f"""
    <div class="soc-header">
        <div class="soc-title">SECURITY OPERATIONS COMMAND // TELEMETRY & INCIDENT RESPONSE</div>
        <div class="soc-badge {status_badge}">{status_label}</div>
    </div>
    """, unsafe_allow_html=True)

    # Telemetry Metric Strip
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Packets Ingested", f"{len(raw_df):,}")
    m2.metric("Unique Origin Nodes", f"{raw_df['Source IP'].nunique():,}")
    current_pps = int(packet_rates.iloc[-1]['Packet Rate']) if not packet_rates.empty else 0
    m3.metric("Current Telemetry Rate", f"{current_pps} PPS", delta=f"Threshold: {ddos_threshold} PPS", delta_color="inverse" if current_pps > ddos_threshold else "normal")
    m4.metric("Mitigation Policy", "CONTAINMENT ENGAGED" if is_attack and "Active" in defense_mode else "AUDIT ACTIVE")

    if is_attack:
        st.markdown(f"<div class='status-strip-red'>VOLUMETRIC FLOOD DETECTED // PRIMARY INGRESS IDENTIFIED: {top_attacker_ip}</div>", unsafe_allow_html=True)
    else:
        st.markdown("<div class='status-strip-green'>NETWORK TRAFFIC NORMAL // OPERATING WITHIN TELEMETRY LIMITS</div>", unsafe_allow_html=True)

    # Telemetry Waveform Plot
    st.markdown("<div class='panel-header'>Real-Time Ingress Volume (Packets / Second)</div>", unsafe_allow_html=True)
    fig = go.Figure()
    plot_series = packet_rates.tail(60) if len(packet_rates) > 60 else packet_rates
    fig.add_trace(go.Scatter(
        x=plot_series['Timestamp_dt'], y=plot_series['Packet Rate'],
        mode='lines', line=dict(color='#38bdf8', width=2.5, shape='spline', smoothing=0.5),
        fill='tozeroy', fillcolor='rgba(56, 189, 248, 0.08)',
        name='Ingress Traffic'
    ))
    fig.add_hline(y=ddos_threshold, line=dict(color='#f43f5e', dash='dash', width=1.2),
                  annotation_text=f"LIMIT {ddos_threshold} PPS", annotation_position="top right")
    fig.update_layout(
        height=240, margin=dict(l=10, r=10, t=10, b=10),
        paper_bgcolor='#0b111c', plot_bgcolor='#0b111c',
        font=dict(color='#94a3b8', family='JetBrains Mono', size=11),
        xaxis=dict(showgrid=True, gridcolor='#182337', zeroline=False),
        yaxis=dict(showgrid=True, gridcolor='#182337', zeroline=False, title='PPS')
    )
    st.plotly_chart(fig, use_container_width=True, config={'displayModeBar': False})

    # Operations Split
    col_left, col_right = st.columns([1.1, 1])

    with col_left:
        st.markdown("<div class='panel-header'>Origin Ingress Matrix (Top Talkers)</div>", unsafe_allow_html=True)
        st.dataframe(top_sources.head(8), use_container_width=True, height=220, hide_index=True)

        st.markdown("<div class='panel-header'>Automated Firewall Audit Trail</div>", unsafe_allow_html=True)
        if st.session_state.mitigation_log:
            st.dataframe(pd.DataFrame(st.session_state.mitigation_log), use_container_width=True, height=130, hide_index=True)
        else:
            st.markdown("<div class='soc-term' style='color:#64748b;'>No active host drops applied. Defense monitoring...</div>", unsafe_allow_html=True)

    with col_right:
        st.markdown("<div class='panel-header'>Active Host Reconnaissance</div>", unsafe_allow_html=True)

        sorted_ips = [ip for ip in top_sources['Source IP'].tolist() if ip != '0.0.0.0']
        selected_ip = st.selectbox("Target Node:", sorted_ips, index=0 if sorted_ips else 0)

        recon_profile = st.radio("Profile:", ["Fast Sweep (-Pn -F -T4)", "Deep OS & Service (-Pn -sV -O --host-timeout 10s -T4)"], horizontal=True)
        flags = "-Pn -F -T4" if "Fast" in recon_profile else "-Pn -sV -O --host-timeout 10s -T4"

        st.markdown(f"<span style='font-family:JetBrains Mono; font-size:11px; color:#64748b;'>Execution Command: <code>sudo nmap {flags} {selected_ip}</code></span>", unsafe_allow_html=True)
        if st.button("Initiate Active Scan", use_container_width=True):
            with st.spinner(f"Executing: sudo nmap {flags} {selected_ip}"):
                cmd, out = run_direct_nmap(selected_ip, flags)
                st.session_state.nmap_output = (cmd, out)

        if st.session_state.nmap_output:
            cmd, out = st.session_state.nmap_output
            st.markdown(f"""
            <div class="soc-term">
            <span style="color:#38bdf8;">$ {cmd}</span><br>
            ----------------------------------------------------------------------<br>
            {out}
            </div>
            """, unsafe_allow_html=True)
            st.markdown(generate_ai_threat_summary(out), unsafe_allow_html=True)

    # Dissector Stream
    st.markdown("<div class='panel-header'>Network Packet Dissection Stream</div>", unsafe_allow_html=True)
    disp_cols = ['Packet ID', 'Time', 'Source IP', 'Destination IP', 'Protocol', 'Length', 'Summary']
    st.dataframe(raw_df[disp_cols].tail(100).iloc[::-1], use_container_width=True, height=200, hide_index=True)

    p_inspect = st.number_input("Inspect Packet ID:", min_value=1, max_value=int(raw_df['Packet ID'].max()), value=int(raw_df['Packet ID'].max()))
    inspect_row = raw_df[raw_df['Packet ID'] == p_inspect]
    if not inspect_row.empty:
        p = inspect_row.iloc[0]
        pl, pr = st.columns(2)
        with pl:
            st.markdown(f"""
            <div class="soc-term">
            <b>[FRAME #{p['Packet ID']} DISSECTION DATA]</b><br>
            Timestamp    : {p['Time']}<br>
            Transport    : {p['Protocol']} (Port {p['Source Port']} -> {p['Destination Port']})<br>
            Ingress IP   : {p['Source IP']} ({p['Location']})<br>
            Target IP    : {p['Destination IP']}<br>
            Frame / TTL  : {p['Length']} Bytes / TTL {p['TTL']}
            </div>
            """, unsafe_allow_html=True)
        with pr:
            st.markdown(f"""
            <div class="soc-term soc-term-amber">
            <b>[HEX DUMP]</b><br>{p['Hex']}<br><br>
            <b>[ASCII DECODE]</b><br>{p['Ascii']}
            </div>
            """, unsafe_allow_html=True)

else:
    if mode == "Upload Forensic PCAP":
        st.info("Forensic Engine Ready. Please upload a .pcap or .pcapng file via the left sidebar.")
    else:
        st.info("System Ready. Click 'Start Acquisition' above to begin capturing telemetry.")

# Live loop refresh while acquisition is running
if st.session_state.is_capturing:
    time.sleep(1.0)
    st.rerun()

Overwriting streamlit_app.py


In [ ]:
# [CELL 4] START STREAMLIT & TUNNEL
import os, subprocess, time
from pyngrok import ngrok
from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

if not NGROK_AUTH_TOKEN:
    raise ValueError("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")

if GOOGLE_API_KEY:
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

# Clean up any lingering processes
subprocess.run("pkill -f streamlit", shell=True)
ngrok.kill()
time.sleep(1)

# Connect tunnel
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(addr='8501', proto='http')
print(f"\n🚀 YOUR LIVE DASHBOARD LINK: {public_url}\n")

# Run Streamlit in background
get_ipython().system_raw('GOOGLE_API_KEY="$GOOGLE_API_KEY" streamlit run streamlit_app.py --server.port 8501 --server.maxUploadSize 500 &>/dev/null&')


🚀 YOUR LIVE DASHBOARD LINK: NgrokTunnel: "https://subsector-domestic-truce.ngrok-free.dev" -> "http://localhost:8501"

